# 02 — Evaluation harness

**Phase 2 deliverable:** `NaiveLag` scored on all three tickers, walk-forward, on a real results
table. This is the project's spine — everything later plugs into it.

The harness was built **before any model**, which is the single most important ordering decision
in the project. A faithful port of a leaking pipeline is just a faster leaking pipeline.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("stock-retrofit @", ROOT)

stock-retrofit @ /Users/kbstudio/Library/CloudStorage/OneDrive-Personal/Project/model/stock-retrofit


## Walk-forward splits

No single fixed tail split anywhere (spec R6). Upstream took the last 30 rows once, *after*
scaling the whole series.

In [2]:
from stock_retrofit.config import EvalConfig
from stock_retrofit.data import load

eval_cfg = EvalConfig.load()
df = load("KBANK")
folds = eval_cfg.splitter().split_frame(df)

print(f"{len(folds)} folds over {len(df)} bars\n")
for f in folds:
    print(" ", f.describe(df["date"]))

8 folds over 6597 bars

  fold 0: train 2021-06-18..2024-07-19 (750) | test 2024-07-23..2024-10-17 (60)
  fold 1: train 2021-09-15..2024-10-17 (750) | test 2024-10-18..2025-01-16 (60)
  fold 2: train 2021-12-15..2025-01-16 (750) | test 2025-01-17..2025-04-16 (60)
  fold 3: train 2022-03-14..2025-04-16 (750) | test 2025-04-17..2025-07-17 (60)
  fold 4: train 2022-06-16..2025-07-17 (750) | test 2025-07-18..2025-10-15 (60)
  fold 5: train 2022-09-14..2025-10-15 (750) | test 2025-10-16..2026-01-15 (60)
  fold 6: train 2022-12-14..2026-01-15 (750) | test 2026-01-16..2026-04-16 (60)
  fold 7: train 2023-03-10..2026-04-16 (750) | test 2026-04-17..2026-07-09 (60)


## The leakage guard

Intending to fit inside folds is not enough — the upstream bug is invisible in the output, since
the numbers simply come out better than they should. So every statistic that learns anything
registers the rows it saw, and the guard raises if any of them belongs to the test block.

In [3]:
import numpy as np
from stock_retrofit.eval.leakage import LeakageError, leakage_guard, register_fit

# Correct: the fit stops exactly at the fold boundary.
with leakage_guard(test_indices=range(900, 960), fold_index=0):
    register_fit("scaler", np.arange(0, 900))
print("fit confined to training rows: allowed")

# The upstream ordering: fit the scaler on everything, split afterwards.
try:
    with leakage_guard(test_indices=range(900, 960), fold_index=0):
        register_fit("MinMaxScaler(full series)", np.arange(0, 960))
except LeakageError as exc:
    print("fit on the full series:", exc)

fit confined to training rows: allowed
fit on the full series: fold 0: 'MinMaxScaler(full series)' was fit on 60 test-block row(s) (positions [900, 901, 902, 903, 904]...). Fit every statistic on the training block only.


`tests/test_no_leakage.py` asserts exactly this, and the guard was verified by *reintroducing*
the upstream bug into `prepare_fold` and confirming the suite goes red — then removing it.

## Why the upstream metric had to go

`calculate_accuracy = 1 − sqrt(mean(((real − predict)/real)²))` on **price levels**. On a
near-random-walk series, "tomorrow's price is today's" scores in the high nineties. This is why
the upstream README's numbers look impressive.

In [4]:
from stock_retrofit.eval.metrics import mase, upstream_accuracy_do_not_use

close = df["close"].to_numpy()
real, lag = close[1:], close[:-1]
print(f"upstream accuracy of a pure lag on KBANK prices : {upstream_accuracy_do_not_use(real, lag):.4f}")

returns = real / lag - 1.0
print(f"MASE of the same lag on returns                  : {mase(returns, np.zeros_like(returns)):.4f}")
print("\n99%+ 'accuracy' and zero skill are the same forecast, scored two ways.")

upstream accuracy of a pure lag on KBANK prices : 0.9798
MASE of the same lag on returns                  : 1.0000

99%+ 'accuracy' and zero skill are the same forecast, scored two ways.


## The baseline, scored on all three tickers

`NaiveLag` is registered like any other model and appears on every results table automatically
(spec R8). Its MASE is 1.0 by construction — that is the line every other model is measured
against. It abstains from directional calls, so its accuracy is undefined rather than 0%.

In [5]:
from stock_retrofit.config import MarketConfigSpec
from stock_retrofit.eval import render_table, results_table, run_walk_forward
from stock_retrofit.models import build

market = MarketConfigSpec.load()
results = []
for symbol in ["KBANK", "SCB", "BAY"]:
    frame = load(symbol)
    for kind in ["naive_lag", "drift", "momentum"]:
        results.append(run_walk_forward(
            build(kind, name=kind), frame,
            splitter=eval_cfg.splitter(), window=eval_cfg.window(),
            symbol=symbol, seed=eval_cfg.seed,
            cost_per_turn=market.round_trip_cost,
        ))

for symbol in ["KBANK", "SCB", "BAY"]:
    subset = [r for r in results if r.symbol == symbol]
    print(render_table(results_table(subset, cost_per_turn=market.round_trip_cost),
                       title=f"{symbol} — baselines"))
    print()

KBANK — baselines
    model symbol  folds   n     ic ic_t   MASE dir_acc RMSE_ret sharpe_net sharpe_gross turnover
naive_lag  KBANK      8 439      —    — 1.0000       —  0.01347      +0.00        +0.00     0.00
    drift  KBANK      8 439 +0.047 +1.0 1.0010   53.9%  0.01344      +1.69        +1.70     0.00
 momentum  KBANK      8 439 -0.023 -0.5 1.4944   49.5%  0.01908      -1.29        +1.22     0.48

IC (forecast vs realised return): mean +0.012 over 2 models, 1 positive, 0 with |t| > 1.96 (~0 expected by chance).
Reference lines: naive_lag Sharpe +0.00 — 0 of 2 models beat holding the share.

SCB — baselines
    model symbol  folds   n     ic ic_t   MASE dir_acc RMSE_ret sharpe_net sharpe_gross turnover
naive_lag    SCB      4 237      —    — 1.0000       —  0.01011      +0.00        +0.00     0.00
    drift    SCB      4 237 -0.094 -1.4 1.0059   53.7%  0.01011      +0.94        +0.97     0.00
 momentum    SCB      4 237 -0.102 -1.6 1.6174   41.3%  0.01520      -3.82        -0.44  